# Reproduce the paper figures

This notebook regenerates **Figure 1** of the paper — the two panels comparing
dimension-reduced GKP-LDLC codes to concatenated GKP–surface-code baselines:

* **(a)** the three logical distances $\\Delta_X,\\Delta_Y,\\Delta_Z$ of every
  reduced instance, from the frozen instance set in `data/paper/`;
* **(b)** the logical error probability $P_L(\\sigma)$ of matched LDLC / surface-code
  pairs, from `data/paper/voronoi_sigma_fine.csv`.

Both panels are written as vector **PDF** with the exact styling used in the paper
(`distance_plot_panel.pdf`, `voronoi_measure_panel.pdf`). Set `FIGDIR` below to the
paper's `Figures/` directory to regenerate them in place.

**Data.** The 60 reduced instances behind panel (a) ship zipped in
`data/paper/instances.zip`; the setup cell extracts them to `data/paper/instances/`
on first run (that folder is git-ignored). This is the *frozen* data that produced
the published figures — distinct from `data/generated/`, the scratch dir where the
generation scripts write new instances.

Run this notebook with the lightweight `examples` environment (no LatticeDecoder /
Oscar needed):

```
julia --project=examples -e 'using Pkg; Pkg.instantiate()'
```


## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)            # the lightweight examples/ environment
using JLD2, Plots, Measures, LaTeXStrings, ZipFile
gr()

EXAMPLES_DIR = @__DIR__
PKG_DIR      = dirname(EXAMPLES_DIR)
PAPER_DIR    = joinpath(PKG_DIR, "data", "paper")           # frozen data behind the paper figures
OUTDIR       = joinpath(PAPER_DIR, "instances")             # reduced instances (JLD2), extracted below
CSV_PATH     = joinpath(PAPER_DIR, "voronoi_sigma_fine.csv")  # P_L Monte-Carlo results (Fig 1b)

# The instances ship zipped (data/paper/instances.zip); extract them once.
if !isdir(OUTDIR) || isempty(filter(f -> endswith(f, ".jld2"), readdir(OUTDIR)))
    mkpath(OUTDIR)
    zpath = joinpath(PAPER_DIR, "instances.zip")
    isfile(zpath) || error("paper instance archive not found: $zpath")
    r = ZipFile.Reader(zpath)
    for f in r.files
        endswith(f.name, ".jld2") || continue
        write(joinpath(OUTDIR, basename(f.name)), read(f))
    end
    close(r)
    @info "extracted paper instances" OUTDIR n=length(filter(f -> endswith(f, ".jld2"), readdir(OUTDIR)))
end

# Where the paper-ready PDFs are written. Point this at the paper's Figures/ folder
# to regenerate the figures in place, e.g.
#   FIGDIR = "/path/to/paper_quantum_ldlc/Figures"
FIGDIR = EXAMPLES_DIR

isfile(CSV_PATH) || @warn "results CSV not found at $CSV_PATH"
@info "figures will be written to" FIGDIR

## Figure 1(a) — distances of the reduced GKP-LDLC codes

Reads `dX`, `dY`, `dZ` from every `reduced_ldlc_gkp_n_<n>_<id>.jld2` whose distances
have been computed, staggers the instances horizontally within each mode-number group,
and overlays the four surface-code / GKP reference distances.

In [ ]:
pattern = r"^reduced_ldlc_gkp_n_(\d+)_(\d+)\.jld2$"
n_values = Int[]; dX_values = Float64[]; dY_values = Float64[]; dZ_values = Float64[]
for file in sort(readdir(OUTDIR))
    m = match(pattern, file); m === nothing && continue
    vals = JLD2.jldopen(joinpath(OUTDIR, file), "r") do io
        status = haskey(io, "distances_status") ? io["distances_status"] : "ok"
        status == "ok" ? (io["dX"], io["dY"], io["dZ"]) : nothing
    end
    vals === nothing && continue      # skip instances whose distances aren't computed yet
    push!(n_values, parse(Int, m.captures[1]))
    push!(dX_values, vals[1]); push!(dY_values, vals[2]); push!(dZ_values, vals[3])
end
isempty(n_values) && error("No completed instances found in $OUTDIR")
@info "loaded $(length(n_values)) reduced instance(s)"

# staggered x-position for each instance within its number-of-modes group
stagger = 2.7 / 30
x_positions = zeros(Float64, length(n_values))
for n in unique(n_values)
    idxs = findall(==(n), n_values); c = length(idxs)
    for (i, idx) in enumerate(idxs)
        x_positions[idx] = n + (i - (c + 0.5) / 2) * stagger
    end
end

Plots.default(size=(780, 520), margins=5mm)
plt_a = scatter(x_positions, dX_values; marker=:circle, ms=5, color=:red, label=L"$\Delta_X$",
                legend=:bottomright, legendfontsize=12, labelfontsize=14, tickfontsize=12,
                legend_columns=1, background_color_legend=RGBA(1,1,1,0.9),
                left_margin=15mm, bottom_margin=7mm, right_margin=3mm, top_margin=5mm)
scatter!(plt_a, x_positions, dY_values; marker=:square,  ms=5, color=:blue,  label=L"$\Delta_Y$")
scatter!(plt_a, x_positions, dZ_values; marker=:diamond, ms=5, color=:green, label=L"$\Delta_Z$")

# light gridline per instance, bold separators between groups
vline!(plt_a, x_positions; color=:gray, alpha=0.3, label="")
vline!(plt_a, [n + 0.5 for n in minimum(n_values):(maximum(n_values) - 1)];
       color=:black, alpha=0.9, lw=1.5, label="")

# surface-code reference distances (square & hexagonal single-mode GKP; d=3/n=9 and d=5/n=25)
sq9  = sqrt(3 / 2);   hex9  = 3^(1 / 4)
sq25 = sqrt(5 / 2);   hex25 = sqrt(5 / sqrt(3))
hline!(plt_a, [sq9];   color=:red,  linestyle=:solid,   lw=1.5, label="Sq-SC (n=9)")
hline!(plt_a, [sq25];  color=:red,  linestyle=:dashdot, lw=1.5, label="Sq-SC (n=25)")
hline!(plt_a, [hex9];  color=:blue, linestyle=:dash,    lw=1.5, label="Hex-SC (n=9)")
hline!(plt_a, [hex25]; color=:blue, linestyle=:dot,     lw=1.5, label="Hex-SC (n=25)")

xlabel!(plt_a, "number of modes"); ylabel!(plt_a, "Distance")
xticks!(plt_a, sort(unique(n_values)))
xlims!(plt_a, minimum(n_values) - 0.5, maximum(n_values) + 0.5)
allv = vcat(dX_values, dY_values, dZ_values); refs = [sq9, sq25, hex9, hex25]
ylims!(plt_a, min(minimum(allv), minimum(refs)) * 0.95, max(maximum(allv), maximum(refs)) * 1.05)

# bold panel label "(a)" outside the plot, to the left of the y axis (top)
(xl, xr) = xlims(plt_a); (yb, yt) = ylims(plt_a)
annotate!(plt_a, xl - 0.10 * (xr - xl), yt, text(L"\mathbf{(a)}", 18, :black, :right, :top))

savefig(plt_a, joinpath(FIGDIR, "distance_plot_panel.pdf"))
@info "saved" joinpath(FIGDIR, "distance_plot_panel.pdf")
plt_a

## Figure 1(b) — logical error probability of the reduced codes

Reads the Monte-Carlo $P_L(\sigma)$ estimates from `voronoi_sigma_fine.csv` and draws the
matched LDLC / hexagonal-GKP-surface-code pairs (one hue per mode count; LDLC solid+filled,
surface dashed+open), with capped binomial error bars. Points with too few samples or logical
events are dropped.

In [ ]:
isfile(CSV_PATH) || error("results CSV not found: $CSV_PATH")
lines = readlines(CSV_PATH)
H = split(lines[1], ','); col(name) = findfirst(==(name), H)
ci = (code=col("code"), n=col("n"), sig=col("sigma"), PL=col("P_L"), se=col("P_L_se"), N=col("nsamples"))
pf(s) = parse(Float64, s)
rows = [split(l, ',') for l in lines[2:end] if !isempty(strip(l))]

MIN_SAMPLES = 20
MIN_EVENTS  = 5      # drop Poisson-dead points (< this many logical-error events)
SIGMA_MAX   = 0.25   # low-noise showcase region

function series(code)
    rs = filter(rows) do r
        r[ci.code] == code || return false
        pf(r[ci.sig]) <= SIGMA_MAX + 1e-9 || return false
        N = parse(Int, r[ci.N]); N >= MIN_SAMPLES || return false
        round(Int, pf(r[ci.PL]) * N) >= MIN_EVENTS
    end
    isempty(rs) && return nothing
    o = sortperm(pf.(getindex.(rs, ci.sig)))
    (σ = pf.(getindex.(rs, ci.sig))[o], PL = pf.(getindex.(rs, ci.PL))[o],
     se = pf.(getindex.(rs, ci.se))[o], n = parse(Int, rs[1][ci.n]))
end

# matched pairs share a hue; LDLC solid/circle, surface dashed/square
c15 = RGB(0.82, 0.10, 0.12); c16 = RGB(0.11, 0.36, 0.79)
c17 = RGB(0.49, 0.17, 0.63); cSC = RGB(0.45, 0.45, 0.45)
SPECS = [
    ("reduced_ldlc_gkp_n_15_5.jld2",  "LDLC n=15",         c15, true,  :circle),
    ("SC_3x5",                        "surface 3×5, n=15", c15, false, :square),
    ("reduced_ldlc_gkp_n_16_8.jld2",  "LDLC n=16",         c16, true,  :circle),
    ("SC_4x4",                        "surface 4×4, n=16", c16, false, :square),
    ("reduced_ldlc_gkp_n_17_15.jld2", "LDLC n=17",         c17, true,  :diamond),
    ("SC_5x5",                        "surface 5×5, n=25", cSC, false, :square),
]

Plots.default(size = (780, 560), margins = 5mm)
plt_b = plot(; yscale = :log10, legend = :bottomright, legendfontsize = 13,
             xguidefontsize = 16, yguidefontsize = 13, tickfontsize = 13,
             background_color_legend = RGBA(1,1,1,0.85),
             xlabel = "Gaussian noise σ", ylabel = L"logical error probability $P_L$",
             left_margin = 15mm, bottom_margin = 8mm, top_margin = 5mm)

los = Float64[]; his = Float64[]; CAPW = 0.0016
for (code, label, c, isldlc, mk) in SPECS
    d = series(code); d === nothing && continue
    lo = max.(d.PL .- d.se, 1e-4); hi = d.PL .+ d.se
    append!(los, lo); append!(his, hi)
    plot!(plt_b, d.σ, d.PL; color = c, lw = 3.3, linestyle = isldlc ? :solid : :dash,
          marker = mk, ms = 5, markerstrokecolor = c,
          markercolor = isldlc ? c : :white, label = label)
    for i in eachindex(d.σ)
        x = d.σ[i]
        plot!(plt_b, [x, x], [lo[i], hi[i]]; color = c, lw = 1.95, label = "")
        plot!(plt_b, [x - CAPW, x + CAPW], [hi[i], hi[i]]; color = c, lw = 1.95, label = "")
        plot!(plt_b, [x - CAPW, x + CAPW], [lo[i], lo[i]]; color = c, lw = 1.95, label = "")
    end
end
ylims!(plt_b, 0.75 * minimum(los), min(1.0, 1.18 * maximum(his)))

# bold panel label "(b)" outside the plot, to the left of the y axis (top)
(xl, xr) = xlims(plt_b); (yb, yt) = ylims(plt_b)
annotate!(plt_b, xl - 0.10 * (xr - xl), yt, text(L"\mathbf{(b)}", 18, :black, :right, :top))

savefig(plt_b, joinpath(FIGDIR, "voronoi_measure_panel.pdf"))
@info "saved" joinpath(FIGDIR, "voronoi_measure_panel.pdf")
plt_b